# Bölüm 11: Transformer'ı oluşturmak

> "Biz tekrar tekrar yaptığımız şeyleriz. Öyleyse mükemmellik bir eylem değil, bir alışkanlıktır."
> — **Aristoteles**, Filozof

---

## Neler öğreneceksin

- Transformer bloklarını nasıl tam bir dil modeline dönüştüreceğiz
- Model deneylemeyi kolaylaştıran yapılandırma kalıbı
- Dil modelleme başının ne yaptığı ve neden buna ihtiyacımız olduğu
- Ağırlık bağlamanın yaklaşık 38 milyon parametreden nasıl tasarruf sağladığı
- Modelini eğitmeden önce doğrulamak için temel tutarlılık kontrolleri
- Önceden eğitilmiş GPT-2 ağırlıklarını kendi mimarine nasıl yükleyeceğin

---

## Kurulum

Önce gerekli paketleri yükleyelim:

In [ ]:
# Gerekli paketleri yükle
!pip install -q torch transformers

In [ ]:
# ===== İÇE AKTARMALAR =====
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from transformers import AutoTokenizer

# Cihazı ayarla
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Kullanılan cihaz: {device}")

## 1. Bölüm 10'dan bileşenler

Önce, Bölüm 10'da oluşturduğumuz bileşenleri getirelim: `MultiHeadAttention`, `FeedForward` ve `TransformerBlock`.

In [ ]:
# ===== ÇOK BAŞLI DİKKAT (Bölüm 10'dan) =====
class MultiHeadAttention(nn.Module):
    """
    Verimli çok başlı dikkat (tüm başları birlikte yığınlar).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Birleşik QKV izdüşümü (ayrı ayrı yapmaktan 3 kat verimli!)
        # Neden 3 * d_model? Çünkü Q, K, V'ye aynı anda izdüşüm yapıyoruz
        # bias=True GPT-2 ile eşleşir; böylece önceden eğitilmiş sapma terimlerini sonra yükleyebiliriz
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Çıktı izdüşümü
        self.out_proj = nn.Linear(d_model, d_model)

        # Dikkat ağırlıklarında dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Adım 1: Q, K, V'ye izdüşüm (hepsi tek seferde!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Adım 2: Q, K, V'ye böl ve çok başlı için yeniden şekillendir =====
        # (batch, seq, 3, num_heads, d_head) şekline yeniden şekillendir
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # (3, batch, num_heads, seq, d_head) düzenine geçir
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Q, K, V'ye böl: her biri (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Adım 3: Ölçeklendirilmiş nokta çarpım dikkati (başlar üzerinde yığınlanmış) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Verildiyse nedensel maskeyi uygula
        if mask is not None:
            # Maskeyi başlar için genişlet: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Değerlerin ağırlıklı toplamı
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Adım 4: Başları birleştir =====
        # (batch, seq, num_heads, d_head) düzenine transpoz et
        attn_output = attn_output.transpose(1, 2)

        # (batch, seq, d_model) şekline yeniden şekillendir; bu başları birleştirir
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Adım 5: Son izdüşüm =====
        output = self.out_proj(attn_output)

        return output, attn_weights




In [ ]:
# ===== İLERİ BESLEMELİ AĞ (Bölüm 10'dan) =====
class FeedForward(nn.Module):
    """
    Konuma göre ileri beslemeli ağ.
    Her konuma bağımsız olarak uygulanır (tüm konumlar için aynı ağırlıklar).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (GPT-2 small için 768)
            d_ff: İleri beslemeli boyutu (tipik olarak 4 * d_model = 3072)
            dropout: Dropout olasılığı
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), genişlet
        x = F.gelu(x, approximate="tanh")  # Doğrusal olmayan (GPT-2'nin GELU varyantı)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), geri izdüşüm
        return x



print("FeedForward tanımlandı!")

In [ ]:
# ===== TRANSFORMER BLOĞU (Bölüm 10'dan) =====
class TransformerBlock(nn.Module):
    """
    Çok başlı dikkat, ileri beslemeli, artık bağlantılar ve
    katman normalizasyonu ile tam Transformer bloğu (GPT-2 gibi ön-norm stili).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Argümanlar:
            d_model: Model boyutu (768)
            num_heads: Dikkat başı sayısı (12)
            d_ff: İleri beslemeli boyutu (3072 = 4 * d_model)
            dropout: Dropout olasılığı
        """
        super().__init__()

        # Katman normalizasyonu (her alt katmandan önce)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Çok başlı dikkat
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # İleri beslemeli ağ
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (her alt katmandan sonra uygulanır)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Argümanlar:
            x: Girdi gömmeleri (batch, seq, d_model)
            mask: Nedensel maske (seq, seq) veya (1, 1, seq, seq)

        Döndürür:
            x: Çıktı gömmeleri (batch, seq, d_model)
            attn_weights: Dikkat ağırlıkları (batch, heads, seq, seq)
        """
        # ===== Artık Bağlantılı Çok Başlı Dikkat =====
        # Ön-norm: Dikkatten ÖNCE normalleştir
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Artık bağlantı

        # ===== Artık Bağlantılı İleri Besleme =====
        # Ön-norm: FFN'den ÖNCE normalleştir
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Artık bağlantı

        return x, attn_weights




## 2. Model yapılandırması

Tüm model hiperparametrelerini bir araya getirmek için bir yapılandırma veri sınıfı oluşturalım.

In [ ]:
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """MiniGPT modeli için yapılandırma."""
    vocab_size: int = 50257      # GPT-2 kelime dağarcığı boyutu
    max_seq_len: int = 1024      # Maksimum bağlam uzunluğu
    embed_dim: int = 768         # Gömme boyutu (d_model)
    num_heads: int = 12          # Dikkat başı sayısı
    num_layers: int = 12         # Transformer bloğu sayısı
    d_ff: int = 3072             # İleri beslemeli gizli boyut (4 * embed_dim)
    dropout: float = 0.1         # Dropout olasılığı

    def __post_init__(self):
        """Başlatmadan sonra yapılandırmayı doğrula."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}), num_heads " \
            f"({self.num_heads}) ile bölünebilir olmalıdır"


## 3. Tam MiniGPT modeli

Şimdi Transformer bloklarını yığınlayarak ve ağırlık bağlaması ile dil modelleme başını ekleyerek tam modeli oluşturalım.

In [ ]:
class MiniGPT(nn.Module):
    """
    Minimal bir GPT tarzı dil modeli.
    Gömmeleri, Transformer bloklarını ve dil modelleme başını birleştirir.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token ve Konum Gömmeleri (Bölüm 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blokları (Bölüm 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Son LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Dil Modelleme Başı =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Ağırlık Bağlama =====
        # Gömme ve çıktı izdüşümü arasında ağırlıkları paylaş
        self.lm_head.weight = self.token_embed.weight

        # Ağırlıkları başlat
        self._init_weights()

    def _init_weights(self):
        """Ağırlıkları küçük rastgele değerlerle başlat."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Model üzerinden ileri geçiş.

        Argümanlar:
            token_ids: Girdi token kimlikleri (batch, seq)
            return_attention: Dikkat ağırlıklarının döndürülüp döndürülmeyeceği

        Döndürür:
            logits: Kelime dağarcığı skorları (batch, seq, vocab_size)
            attention_weights: Blok başına dikkat ağırlıklarının isteğe bağlı listesi
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Gömmeler =====
        # Token gömmeleri
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Konum gömmeleri
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Birleştir ve dropout uygula
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Nedensel Maske Oluştur =====
        # Alt üçgen: her konum yalnızca önceki konumlara dikkat edebilir
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Transformer Bloklarından Geçir =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Son LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Dil Modelleme Başı =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        input_ids'i devam ettiren token'lar üret (prompt çıktıya dahildir).

        Argümanlar:
            input_ids: Prompt’un token kimlikleri (batch, seq)
            max_new_tokens: Kaç yeni token üretilecek
            temperature: Örnekleme sıcaklığı (yalnızca do_sample=True iken kullanılır)
            do_sample: Dağılımdan örnekle; False = açgözlü argmax
            eos_token_id: Her dizi bu token'ı üretirse erken dur

        Döndürür:
            Token kimlikleri (batch, seq + yeni token'lar)
        """
        self.eval()

        # Konum gömmeleri yalnızca max_seq_len konumu kapsar
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} token) + max_new_tokens "
                f"({max_new_tokens}) modelin {limit} token'lık bağlamını "
                "aşıyor. Prompt'u kısalt ya da max_new_tokens'ı düşür.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


In [ ]:
# Minik yapılandırmayla hızlı test
tiny_config = GPTConfig(vocab_size=500, max_seq_len=16, embed_dim=64,
                        num_heads=2, num_layers=2, d_ff=256, dropout=0.0)
model = MiniGPT(tiny_config)

# İleri geçişi test et
test_tokens = torch.randint(0, tiny_config.vocab_size, (2, 16))
logits = model(test_tokens)

print(f"Girdi şekli: {test_tokens.shape}")
print(f"Çıktı şekli: {logits.shape}")
print(f"\nBeklenen: (2, 16, {tiny_config.vocab_size})")

## 4. Tutarlılık kontrolleri

Önce modelin bileşenlerini beş testle doğrula. İleride altıncı testte GPT-2 ağırlıklarını yükledikten sonra iki modelin logitlerini karşılaştıracaksın.

In [ ]:
# ===== TEST 1: Şekil Doğrulama =====

def test_forward_shapes(config):
    """Model çıktı şekillerini doğrula."""
    model = MiniGPT(config)
    
    batch_size, seq_len = 2, 16
    token_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
    
    logits = model(token_ids)
    
    expected_shape = (batch_size, seq_len, config.vocab_size)
    assert logits.shape == expected_shape, \
        f"Beklenen {expected_shape}, alınan {logits.shape}"
    
    print(f"Şekil kontrolü BAŞARILI: {logits.shape}")

test_forward_shapes(tiny_config)

In [ ]:
# ===== TEST 2: Parametre Sayısı =====
def count_parameters(model):
    """Toplam eğitilebilir parametreleri say."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def test_parameter_count(config):
    """Parametre sayısının makul olduğunu doğrula."""
    model = MiniGPT(config)
    total = count_parameters(model)

    # GPT-2 Small'da ~124M parametre var
    print(f"Toplam parametreler: {total:,}")

    # Kesin döküm (sapma terimleri dahil)
    embed_params = config.vocab_size * config.embed_dim  # Token gömmeleri
    pos_params = config.max_seq_len * config.embed_dim   # Konum gömmeleri
    d = config.embed_dim
    block_params = config.num_layers * (
        4 * d * d + 4 * d +                     # Dikkat: QKV + çıktı, sapma terimleriyle
        2 * d * config.d_ff + config.d_ff + d + # FFN: fc1 + fc2, sapma terimleriyle
        4 * d                                   # LayerNorm'lar
    )
    ln_f_params = 2 * d                          # Son LayerNorm
    # Not: lm_head parametreleri token_embed'e bağlıdır, ayrı sayılmaz

    expected = embed_params + pos_params + block_params + ln_f_params
    print(f"Beklenen: {expected:,}")
    assert total == expected, f"Sayım uyuşmazlığı: {total:,} ve {expected:,}"


# Minik yapılandırmayla test et
print("Minik model:")
test_parameter_count(tiny_config)

# Tam GPT-2 yapılandırmasıyla test et
print("\nGPT-2 Small:")
test_parameter_count(GPTConfig())

In [ ]:
# ===== TEST 3: Nedensel Maskeleme =====

def test_causal_masking(config):
    """Modelin gelecek tokenları göremediğini doğrula."""
    model = MiniGPT(config)
    model.eval()

    # Aynı önek, farklı sonek
    tokens_a = torch.tensor([[100, 200, 300, 400]])
    tokens_b = torch.tensor([[100, 200, 300, 401]])  # Son token farklı

    with torch.no_grad():
        logits_a = model(tokens_a)
        logits_b = model(tokens_b)

    # 0, 1, 2 konumları AYNI olmalı
    for pos in range(3):
        assert torch.allclose(logits_a[0, pos], logits_b[0, pos], atol=1e-5), \
            f"Konum {pos} logitleri farklı!"

    # Konum 3 FARKLI olmalı
    assert not torch.allclose(logits_a[0, 3], logits_b[0, 3], atol=1e-5), \
        "Konum 3 logitleri farklı girdiye rağmen aynı!"

    print("Nedensel maskeleme BAŞARILI!")

test_causal_masking(tiny_config)

In [ ]:
# ===== TEST 4: Ağırlık Bağlama =====

def test_weight_tying(config):
    """Gömme ve lm_head'in ağırlıkları paylaştığını doğrula."""
    model = MiniGPT(config)

    # AYNI tensör olmalı
    assert model.lm_head.weight is model.token_embed.weight, \
        "Ağırlık bağlama başarısız: farklı tensörler!"

    # Birini değiştir, diğerinin değiştiğini kontrol et
    with torch.no_grad():
        original = model.token_embed.weight[0, 0].item()
        model.token_embed.weight[0, 0] = 999.0
        
        assert model.lm_head.weight[0, 0].item() == 999.0, \
            "Ağırlık bağlama başarısız: değişiklikler yayılmıyor!"
        
        model.token_embed.weight[0, 0] = original

    print("Ağırlık bağlama BAŞARILI!")

test_weight_tying(tiny_config)

In [ ]:
# ===== TEST 5: Gradyan Akışı =====

def test_gradient_flow(config):
    """Gradyanların tüm parametrelere ulaştığını doğrula."""
    model = MiniGPT(config)

    # İleri geçiş
    tokens = torch.randint(0, config.vocab_size, (1, 8))
    logits = model(tokens)

    # Geri geçiş
    loss = logits.sum()
    loss.backward()

    # Tüm parametrelerin gradyanları olduğunu kontrol et
    for name, param in model.named_parameters():
        assert param.grad is not None, f"{name} için gradyan yok"

    print("Gradyan akışı BAŞARILI!")

test_gradient_flow(tiny_config)

## 5. İlk ileri geçişin

Gerçek metni modelimizden geçirelim ve rastgele ağırlıklarla ne çıktı verdiğini görelim.

In [ ]:
# Tokenizer'ı yükle
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Küçük model oluştur
config = GPTConfig(num_layers=2, embed_dim=256, num_heads=4, d_ff=1024)
model = MiniGPT(config)
model.eval()

# Bir prompt tokenize et
prompt = "The quick brown fox"
token_ids = tokenizer.encode(prompt, return_tensors="pt")

print(f"Prompt: '{prompt}'")
print(f"Token kimlikleri: {token_ids}")
print(f"Tokenlar: {[tokenizer.decode([t]) for t in token_ids[0]]}")

In [ ]:
# İleri geçiş
with torch.no_grad():
    logits = model(token_ids)

print(f"Logit şekli: {logits.shape}")

# Sonraki token için en iyi 5 tahmini al
last_logits = logits[0, -1, :]
top_probs, top_indices = torch.softmax(last_logits, dim=-1).topk(5)

print(f"\n'{prompt}' sonrası en iyi 5 tahmin:")
for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx])
    print(f"  '{token}': {prob:.4f}")

print("\n(Rastgele tahminler - modelin ağırlıkları rastgele!)")

## 6. Önceden eğitilmiş ağırlıkları yükleme

Şimdi heyecan verici kısım: gerçek GPT-2 ağırlıklarını MiniGPT'mize yükleyelim!

In [ ]:
def load_gpt2_weights(model, model_name="gpt2"):
    """
    Önceden eğitilmiş GPT-2 ağırlıklarını MiniGPT modelimize yükle.

    Argümanlar:
        model: MiniGPT örneğimiz
        model_name: HuggingFace model adı ("gpt2", "gpt2-medium" vb.)

    Döndürür:
        ağırlıkları yüklenmiş model
    """
    from transformers import GPT2LMHeadModel

    print(f"'{model_name}' modelinden ağırlıklar yükleniyor...")

    # HuggingFace modelini yükle
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    hf_state = hf_model.state_dict()

    # Bizim modelimizin state dict'i
    our_state = model.state_dict()

    # Gömmeleri kopyala
    our_state['token_embed.weight'].copy_(hf_state['transformer.wte.weight'])
    our_state['pos_embed.weight'].copy_(hf_state['transformer.wpe.weight'])

    # Her Transformer bloğunu kopyala
    for i in range(model.config.num_layers):
        # Layer normları
        our_state[f'blocks.{i}.ln1.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.weight'])
        our_state[f'blocks.{i}.ln1.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_1.bias'])
        our_state[f'blocks.{i}.ln2.weight'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.weight'])
        our_state[f'blocks.{i}.ln2.bias'].copy_(
            hf_state[f'transformer.h.{i}.ln_2.bias'])

        # Dikkat (ağırlıkları transpoz etmek gerekir; sapma terimleri doğrudan kopyalanır)
        our_state[f'blocks.{i}.attn.qkv_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.weight'].T)
        our_state[f'blocks.{i}.attn.qkv_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_attn.bias'])
        our_state[f'blocks.{i}.attn.out_proj.weight'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.weight'].T)
        our_state[f'blocks.{i}.attn.out_proj.bias'].copy_(
            hf_state[f'transformer.h.{i}.attn.c_proj.bias'])

        # FFN (transpoze etmek gerekiyor!)
        our_state[f'blocks.{i}.ffn.fc1.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.weight'].T)
        our_state[f'blocks.{i}.ffn.fc1.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_fc.bias'])
        our_state[f'blocks.{i}.ffn.fc2.weight'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.weight'].T)
        our_state[f'blocks.{i}.ffn.fc2.bias'].copy_(
            hf_state[f'transformer.h.{i}.mlp.c_proj.bias'])

    # Son layer norm
    our_state['ln_f.weight'].copy_(hf_state['transformer.ln_f.weight'])
    our_state['ln_f.bias'].copy_(hf_state['transformer.ln_f.bias'])

    # Not: lm_head.weight, token_embed.weight'e bağlıdır; zaten kopyalandı

    print("Ağırlıklar başarıyla yüklendi!")
    return model


In [ ]:
@torch.no_grad()
def generate_simple(model, tokenizer, prompt, max_new_tokens=20):
    """
    Açgözlü kod çözme kullanarak metin üret (her zaman en olası token'ı seç).

    Argümanlar:
        model: Ağırlıkları yüklenmiş MiniGPT'miz
        tokenizer: GPT-2 tokenizer
        prompt: Başlangıç metni
        max_new_tokens: Kaç yeni token üretilecek

    Döndürür:
        Üretilen metin dizisi
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=False = açgözlü: her adımda argmax
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Tekrar metne kod çöz (prompt dahil)
    return tokenizer.decode(output_ids[0])


In [ ]:
# GPT-2 Small yapılandırmasıyla model oluştur
config = GPTConfig()  # Varsayılanlar GPT-2 Small ile eşleşir
model = MiniGPT(config)

# Önceden eğitilmiş ağırlıkları yükle
model = load_gpt2_weights(model, "gpt2")
model = model.to(device)

print(f"\nModel cihazda: {device}")
print(f"Parametreler: {count_parameters(model):,}")

In [ ]:
# ===== TEST 6: HuggingFace GPT-2 ile logit eşitliği =====
def test_gpt2_parity(model, tokenizer):
    """Yüklediğimiz modelin HuggingFace GPT-2 logitleriyle eşleştiğini doğrula."""
    from transformers import GPT2LMHeadModel

    hf_model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    model.eval()

    # Modelin cihazında karşılaştır (HF modeli de oraya taşınır)
    device = next(model.parameters()).device
    hf_model = hf_model.to(device)
    ids = tokenizer.encode("The quick brown fox jumps over",
                           return_tensors="pt").to(device)
    with torch.no_grad():
        our_logits = model(ids)
        hf_logits = hf_model(ids).logits

    # Minik kayan nokta farkları sorun değil; daha büyüğü bir ağırlığın atlandığı anlamına gelir
    torch.testing.assert_close(our_logits, hf_logits, atol=1e-4, rtol=1e-4)
    print("GPT-2 eşitliği BAŞARILI!")

test_gpt2_parity(model, tokenizer)
# Beklenen: GPT-2 eşitliği BAŞARILI!

In [ ]:
# Metin üret!
prompt = "The quick brown fox"
generated = generate_simple(model, tokenizer, prompt, max_new_tokens=30)

print(f"Prompt: '{prompt}'")
print(f"Üretilen: '{generated}'")

In [ ]:
# Daha fazla prompt dene!
prompts = [
    "Artificial intelligence will",
    "Once upon a time",
    "The capital of France is",
    "def fibonacci(n):"
]

for prompt in prompts:
    generated = generate_simple(model, tokenizer, prompt, max_new_tokens=20)
    print(f"'{prompt}' -> {generated}")
    print()

## 7. Rastgele ve önceden eğitilmiş karşılaştırma

Rastgele ağırlıklar ile önceden eğitilmiş ağırlıkları dramatik bir şekilde karşılaştıralım.

In [ ]:
# Rastgele ağırlıklı model
model_random = MiniGPT(GPTConfig())
model_random = model_random.to(device)

prompt = "Artificial intelligence will"

print("=" * 50)
print("RASTGELE AĞIRLIKLAR:")
print(generate_simple(model_random, tokenizer, prompt, max_new_tokens=20))
print()
print("ÖNCEDEN EĞİTİLMİŞ AĞIRLIKLAR:")
print(generate_simple(model, tokenizer, prompt, max_new_tokens=20))
print("=" * 50)

print("\nAynı mimari. Aynı kod. Eğitim tüm farkı yaratıyor!")

## 8. Alıştırmalar

### Alıştırma 1: minik model

Belirli özelliklere sahip minik bir model oluştur ve test et.

In [ ]:
# KODUNU BURAYA YAZ
# Şu özelliklere sahip minik bir MiniGPT oluştur:
# - vocab_size=500
# - max_seq_len=16
# - embed_dim=64
# - num_heads=2
# - num_layers=2
# - d_ff=256

# 1. Yapılandırmayı oluştur
# 2. Modeli oluştur
# 3. Rastgele tokenlar üzerinde ileri geçiş yap
# 4. Çıktı şeklini doğrula
# 5. Parametreleri say

### Alıştırma 2: sıcaklık keşfi

Sıcaklık ölçeklemeyi kullanmak için üretimi değiştir.

In [ ]:
# KODUNU BURAYA YAZ
# generate_simple'ı bir sıcaklık parametresi kabul edecek şekilde değiştir:
# next_logits = next_logits / temperature
# 
# Şu sıcaklıkları dene: 0.5, 1.0, 1.5
# Çıktı nasıl değişiyor?

### Alıştırma 3: dikkat görselleştirme

Önceden eğitilmiş modeldeki dikkat desenlerini görselleştir.

In [ ]:
# KODUNU BURAYA YAZ
# 1. Modeli return_attention=True ile çalıştır
# 2. Son bloktan dikkat ağırlıklarını al
# 3. Dikkat başı 0 için bir ısı haritası çiz
# İpucu: matplotlib.pyplot.imshow() kullan

### Parametre grupları ön izlemesi (Bölüm 12'de kullanılır)

Bölüm 12, ağırlık azaltmasını yalnızca fayda sağlayan parametrelere uygular. Bu yardımcı o ayrımı uygular: matrisler (2B+) azaltma alır, sapma ve LayerNorm vektörleri almaz.


In [ ]:
def get_parameter_groups(model, weight_decay=0.1):
    """
    Ağırlık azaltması uygulanacak/uygulanmayacak parametreleri ayır.

    Gömme tabloları dahil matrislere ağırlık azaltması uygularız. Tek boyutlu sapma
    ve LayerNorm parametrelerine uygulamayız. Bazı tariflerde gömmeler de dışarıda
    bırakılır; burada her parametre için aynı boyut kuralını kullanıyoruz.
    """
    decay_params = []
    no_decay_params = []

    for name, param in model.named_parameters():
        # Matrisler (2B+) ağırlık azaltması alır; vektörler (sapma, LayerNorm ölçekleri) almaz
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Bölüm 12 optimizasyonunu bu yardımcıyla kurar:
# AdamW(get_parameter_groups(model, weight_decay=0.01), lr=learning_rate)
print("get_parameter_groups tanımlandı!")


## Özet

**Oluşturduklarımız:**

1. **GPTConfig**: Temiz yapılandırma kalıbı
2. **MiniGPT**: Ağırlık bağlamalı tam dil modeli
3. **Tutarlılık kontrolleri**: Doğruluğu kontrol etmek için 6 test (HuggingFace GPT-2 ile logit eşitliği dahil)
4. **Ağırlık yükleme**: GPT-2 ağırlıklarını aktarma
5. **Metin üretimi**: Açgözlü kod çözme

**Temel kavramlar:**

- Katmanları yığınlamak için `nn.ModuleList`
- Ağırlık bağlama 38M parametre tasarrufu sağlar
- LM başı: `(batch, seq, embed_dim)` → `(batch, seq, vocab_size)`
- `model.train()` ve `model.eval()`
- Ağırlık bağlama tuzağı: `'lm_head.weight'` `state_dict().keys()` içinde GÖRÜNÜR; bağlı ağırlıklar aynı depolamayı paylaşır, `torch.save` onu bir kez yazar ve dosya küçük kalır

**Sonraki:** Bölüm 12, bu modeli sıfırdan eğitmeyi öğretecek!